# LSTM-Autoencoder Feature Selection Comparison
This notebook compares three feature selection algorithms:
1. **Mutual Information** (Filter-based)
2. **Random Forest Feature Importance** (Embedded)
3. **Recursive Feature Elimination** (Wrapper-based)

Dataset: CSE-CICIDS2018

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, precision_recall_curve, auc
)
import matplotlib.pyplot as plt
import time
import warnings

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# --- Configuration ---
DATASET_DIR = "/home/dan/AnomalyIDS/dataset"
TIME_STEPS = 10
HIDDEN_DIM = 16
LEARNING_RATE = 0.001
DROPOUT = 0.2
BATCH_SIZE = 64
EPOCHS = 30
K_FEATURES = 10  # Number of features to select for comparison
BENIGN_TRAIN_FRACTION = 0.7  # Fraction of benign data per file for training

In [ ]:
def load_and_split_datasets(dataset_dir):
    all_train_benign = []
    all_test_data = []
    all_test_labels = []
    
    csv_files = sorted(glob.glob(os.path.join(dataset_dir, "*.csv")))
    print(f"Found {len(csv_files)} files.")
    
    for file in csv_files:
        print(f"Processing {os.path.basename(file)}...")
        df = pd.read_csv(file, low_memory=False)
        
        # Basic Cleaning
        # Identify numeric columns (excluding Label)
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        df.dropna(subset=numeric_cols + ['Label'], inplace=True)
        
        # Split Benign vs Attack
        benign = df[df['Label'] == 'Benign'].copy()
        attack = df[df['Label'] != 'Benign'].copy()
        
        # Split Benign into train/test
        split_idx = int(len(benign) * BENIGN_TRAIN_FRACTION)
        train_benign = benign.iloc[:split_idx]
        test_benign = benign.iloc[split_idx:]
        
        all_train_benign.append(train_benign)
        
        # Test set consists of remaining benign + all attacks
        test_combined = pd.concat([test_benign, attack], ignore_index=True)
        all_test_data.append(test_combined)
        
    train_df = pd.concat(all_train_benign, ignore_index=True)
    test_df = pd.concat(all_test_data, ignore_index=True)
    
    # Separate labels and features
    X_train = train_df.drop(columns=['Label'])
    y_train = (train_df['Label'] != 'Benign').astype(int).values # Should be all 0
    
    X_test = test_df.drop(columns=['Label'])
    y_test = (test_df['Label'] != 'Benign').astype(int).values
    
    return X_train, y_train, X_test, y_test

X_train_raw, y_train_raw, X_test_raw, y_test_raw = load_and_split_datasets(DATASET_DIR)

In [ ]:
def select_features_mi(X, y, k):
    mi_scores = mutual_info_classif(X, y, random_state=SEED)
    top_k_idx = np.argsort(mi_scores)[-k:]
    return X.columns[top_k_idx].tolist()

def select_features_rf(X, y, k):
    rf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
    rf.fit(X, y)
    importances = rf.feature_importances_
    top_k_idx = np.argsort(importances)[-k:]
    return X.columns[top_k_idx].tolist()

def select_features_rfe(X, y, k):
    rf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
    rfe = RFE(estimator=rf, n_features_to_select=k, step=1)
    rfe.fit(X, y)
    return X.columns[rfe.support_].tolist()

methods = {
    "Mutual Information": select_features_mi,
    "RF Importance": select_features_rf,
    "RFE": select_features_rfe
}

In [ ]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, num_features, time_steps, hidden_dim=16):
        super(LSTMAutoencoder, self).__init__()
        self.time_steps = time_steps
        self.num_features = num_features
        
        # Encoder
        self.encoder_lstm = nn.LSTM(num_features, hidden_dim, batch_first=True)
        self.encoder_dropout = nn.Dropout(0.2)
        
        # Decoder
        self.decoder_lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.decoder_dropout = nn.Dropout(0.2)
        
        self.time_distributed = nn.Linear(hidden_dim, num_features)

    def forward(self, x):
        _, (hn, _) = self.encoder_lstm(x)
        latent = self.encoder_dropout(hn[-1])
        
        x_repeated = latent.unsqueeze(1).repeat(1, self.time_steps, 1)
        
        decoder_out, _ = self.decoder_lstm(x_repeated)
        decoder_out = self.decoder_dropout(decoder_out)
        
        reconstructed = self.time_distributed(decoder_out)
        return reconstructed

In [ ]:
def create_sequences(data, seq_len):
    sequences = []
    for i in range(0, len(data) - seq_len + 1, seq_len):
        sequences.append(data[i : i + seq_len])
    return np.array(sequences)

def create_sequences_with_labels(data, labels, seq_len):
    sequences = []
    seq_labels = []
    for i in range(0, len(data) - seq_len + 1, seq_len):
        sequences.append(data[i : i + seq_len])
        seq_labels.append(1 if labels[i : i + seq_len].sum() > 0 else 0)
    return np.array(sequences), np.array(seq_labels)

In [ ]:
def train_and_evaluate(method_name, selected_features, X_train_raw, X_test_raw, y_test_raw):
    print(f"\n>>> Evaluating Method: {method_name}")
    print(f"Features: {selected_features}")
    
    # Scale and filter features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw[selected_features].values).astype(np.float32)
    X_test_scaled = scaler.transform(X_test_raw[selected_features].values).astype(np.float32)
    
    # Create sequences
    X_train_seq = create_sequences(X_train_scaled, TIME_STEPS)
    X_test_seq, y_test_seq = create_sequences_with_labels(X_test_scaled, y_test_raw, TIME_STEPS)
    
    # DataLoader
    train_tensor = torch.from_numpy(X_train_seq)
    train_loader = DataLoader(
        TensorDataset(train_tensor, train_tensor),
        batch_size=BATCH_SIZE, shuffle=True, drop_last=True
    )
    
    # Model
    model = LSTMAutoencoder(len(selected_features), TIME_STEPS, HIDDEN_DIM).to(device)
    criterion = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # Training Loop
    model.train()
    for epoch in range(1, EPOCHS + 1):
        total_loss = 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            recon = model(batch_x)
            loss = criterion(recon, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * batch_x.size(0)
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch}/{EPOCHS} | Loss: {total_loss/len(train_loader.dataset):.6f}")
    
    # Determine Threshold (MAX training error)
    model.eval()
    train_errors = []
    with torch.no_grad():
        for batch_x, _ in train_loader:
            batch_x = batch_x.to(device)
            recon = model(batch_x)
            mse = ((batch_x - recon)**2).mean(dim=(1, 2))
            train_errors.append(mse.cpu().numpy())
    
    threshold = np.max(np.concatenate(train_errors))
    print(f"Threshold (Max Train Error): {threshold:.6f}")
    
    # Evaluation
    test_tensor = torch.from_numpy(X_test_seq)
    test_loader = DataLoader(TensorDataset(test_tensor, test_tensor), batch_size=BATCH_SIZE, shuffle=False)
    
    test_errors = []
    with torch.no_grad():
        for batch_x, _ in test_loader:
            batch_x = batch_x.to(device)
            recon = model(batch_x)
            mse = ((batch_x - recon)**2).mean(dim=(1, 2))
            test_errors.append(mse.cpu().numpy())
    
    test_errors = np.concatenate(test_errors)
    y_true = y_test_seq[:len(test_errors)]
    y_pred = (test_errors > threshold).astype(int)
    
    # Metrics
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, test_errors),
    }
    
    # PR-AUC
    prec, rec, _ = precision_recall_curve(y_true, test_errors)
    metrics["PR-AUC"] = auc(rec, prec)
    
    return metrics

In [ ]:
final_results = {}

for name, func in methods.items():
    # Since MI/RF/RFE need labels, we can use a subset of data or all for selection
    # For selection, we use the combined X_train_raw and y_train_raw (even if y is mostly 0)
    # Actually, we should use a mix of Benign and Attack to identify which features distinguish them
    # Let's use a balanced sample from X_train_raw (benign) and X_test_raw (some attacks)
    
    # Sampling for feature selection to avoid huge memory usage
    X_sel = pd.concat([X_train_raw.sample(min(100000, len(X_train_raw))), X_test_raw.sample(min(100000, len(X_test_raw)))])
    y_sel = pd.concat([pd.Series(y_train_raw[:len(X_train_raw.sample(min(100000, len(X_train_raw))))]), 
                       pd.Series(y_test_raw[:len(X_test_raw.sample(min(100000, len(X_test_raw)))]))])
    # Wait, sampling needs to be aligned. Let's do it properly:
    
    # Proper balanced sampling for feature selection
    # We need some attack data to know what to select
    # X_test_raw contains attacks
    attack_indices = np.where(y_test_raw == 1)[0]
    benign_indices = np.where(y_test_raw == 0)[0]
    
    sample_size = 50000
    idx_a = np.random.choice(attack_indices, min(sample_size, len(attack_indices)), replace=False)
    idx_b = np.random.choice(benign_indices, min(sample_size, len(benign_indices)), replace=False)
    
    X_fs = pd.concat([X_test_raw.iloc[idx_a], X_test_raw.iloc[idx_b]])
    y_fs = np.concatenate([np.ones(len(idx_a)), np.zeros(len(idx_b))])
    
    selected_features = func(X_fs, y_fs, K_FEATURES)
    res = train_and_evaluate(name, selected_features, X_train_raw, X_test_raw, y_test_raw)
    final_results[name] = res

# Comparison Table
results_df = pd.DataFrame(final_results).T
print("\n=== FINAL COMPARISON ===")
print(results_df)

In [ ]:
# Plotting the results
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]
results_df[metrics_to_plot].plot(kind='bar', figsize=(12, 6))
plt.title("Comparison of Feature Selection Methods for LSTM-Autoencoder")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()